# 혜진 v9: v6 앙상블 기여도·모델 제거·가중치 비교
새 유전자/파생변수/학습기를 추가하지 않습니다. 현재 v6의 8개 구성 모델을 고정하고 구성 제거와 가중치만 비교합니다. 원본 CV 중심 후보와 기존 스트레스 종합 기준 후보를 최대 2개 고정한 뒤, seed=109/211 그룹 3-fold에서 재학습해 확인합니다. 두 seed도 같은 개발 환자이며 새 독립 환자가 아닙니다.

노트북 하나에 EDA·전처리·학습 내 선별·구성 모델 학습·비교·저장·보고서를 포함했습니다. 프로젝트 Python 파일이나 이전 피클이 필수인 구조가 아닙니다. 선택적 캐시가 없거나 CACHE_COMPONENTS=False면 모든 구성을 다시 학습합니다. 원본 data/와 requirements-v9.txt 환경은 필요합니다. 기존 CV/홀드아웃은 여러 버전에서 관측해 선택 편향이 있습니다. Public 향상을 보장하지 않습니다.

탐색 기록: 초기 40개 단독/그룹 변경에서 충분한 개선이 없어, 개발 CV를 본 뒤 Dirichlet 중심 W6·농도 24/64·고정 seed=904로 64개 공동 가중치 계획을 추가했습니다. 홀수 샘플은 0.03 미만의 가중치를 0으로 한 뒤 정규화합니다. 반복 확인 결과를 보기 전에 총 104개 계획과 선택 규칙을 고정했으며, 적응 탐색에 따른 선택 편향이 있습니다. Run All은 모두 평가합니다.

In [1]:
import os,sys,re,json,pickle,cloudpickle,hashlib,platform,time,copy,warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.special import softmax
from sklearn.base import BaseEstimator,TransformerMixin,clone
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_selection import chi2,f_classif
from sklearn.preprocessing import StandardScaler,MaxAbsScaler
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import f1_score,accuracy_score,classification_report,confusion_matrix
from lightgbm import LGBMClassifier
from threadpoolctl import threadpool_limits
from IPython.display import display,FileLink
threadpool_limits(2)
SEED=42
roots=[Path.cwd(),*Path.cwd().parents]
REPO=next((p for p in roots if all((p/'data'/f).is_file() for f in ['train.csv','test.csv','sample_submission.csv'])),None)
if REPO is None:raise FileNotFoundError('공用 data/에 train.csv, test.csv, sample_submission.csv가 필요합니다.')
DATA=REPO/'data';WORK=REPO/'models'/'혜진' if (REPO/'models'/'혜진').is_dir() else REPO
OUT=WORK/'artifacts'/'v9';OUT.mkdir(parents=True,exist_ok=True)
SUBMIT=REPO/'submission';SUBMIT.mkdir(exist_ok=True)
os.environ['MPLCONFIGDIR']=str(OUT/'.mplconfig')
train=pd.read_csv(DATA/'train.csv');test=pd.read_csv(DATA/'test.csv');sample=pd.read_csv(DATA/'sample_submission.csv')
genes=[c for c in train if c not in ['ID','SUBCLASS']]
assert list(test)==['ID']+genes and sample.ID.equals(test.ID)
assert train.ID.is_unique and test.ID.is_unique
y=train.SUBCLASS.to_numpy();labels=np.sort(np.unique(y));yi=np.searchsorted(labels,y);NC=len(labels)
X=train[genes];groups=pd.util.hash_pandas_object(X,index=False).to_numpy()
dev,hold=next(StratifiedGroupKFold(5,shuffle=True,random_state=SEED).split(X,y,groups))
folds=list(StratifiedGroupKFold(3,shuffle=True,random_state=SEED+1).split(X.iloc[dev],y[dev],groups[dev]))
assert not set(groups[dev]).intersection(groups[hold])
for a,b in folds:assert not set(groups[dev[a]]).intersection(groups[dev[b]])
pd.DataFrame({'ID':train.ID,'split':np.where(np.isin(np.arange(len(train)),hold),'holdout','development'),'group':groups.astype(str)}).to_csv(OUT/'split_manifest.csv',index=False)
print('DATA',DATA,'OUTPUT',OUT,'Train/test',train.shape,test.shape,'Dev/holdout',len(dev),len(hold))

from sklearn.neural_network import MLPClassifier
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier


DATA /Users/mac/project/cancer_project/data OUTPUT /Users/mac/project/cancer_project/artifacts/v9 Train/test (6201, 4386) (2546, 4385) Dev/holdout 4960 1241


In [2]:
# Descriptive EDA only: no test-derived vocabularies or fitted preprocessing.
burden_train=(train[genes].notna()&train[genes].ne('WT')).sum(axis=1)
burden_test=(test[genes].notna()&test[genes].ne('WT')).sum(axis=1)
eda=pd.DataFrame({'train':burden_train.describe(),'test':burden_test.describe()})
eda.to_csv(OUT/'eda_mutated_gene_burden.csv')
byclass=pd.DataFrame({'SUBCLASS':y,'mutated_genes':burden_train}).groupby('SUBCLASS').mutated_genes.agg(['count','mean','median','max'])
byclass.to_csv(OUT/'eda_by_class.csv')
missing=pd.DataFrame({'train_missing':train[genes].isna().sum(),'test_missing':test[genes].isna().sum()})
missing.to_csv(OUT/'eda_missing_by_gene.csv')
print('4384 original gene columns; mutation burden is the number of non-WT non-missing gene cells, not the token count.')
display(eda);display(byclass)


4384 original gene columns; mutation burden is the number of non-WT non-missing gene cells, not the token count.


,train,test
count,6201.000000,2546.000000
mean,35.299629,78.134328
std,100.776379,213.964535
min,0.000000,0.000000
25%,7.000000,11.000000
50%,14.000000,28.000000
75%,27.000000,52.000000
max,2393.000000,2784.000000


,count,mean,median,max
SUBCLASS,,,,
ACC,72,39.430556,24.0,396
BLCA,104,70.490385,54.0,382
BRCA,786,17.437659,9.0,853
CESC,155,44.129032,20.0,480
COAD,223,73.121076,23.0,1631
DLBC,38,67.421053,30.0,704
GBMLGG,461,12.535792,11.0,121
HNSC,223,40.569507,31.0,386
KIPAN,515,16.042718,15.0,246


In [3]:
GROUPS = {
 'p53_cellcycle': 'TP53 RB1 CDKN2A CDKN2B CDK4 CDK6 CCND1 ATM CHEK2'.split(),
 'rtk_ras_mapk': 'EGFR ERBB2 ERBB3 KRAS NRAS HRAS BRAF NF1 MAP2K1 MAP2K2 KIT PDGFRA FGFR2 FGFR3'.split(),
 'pi3k_mtor': 'PIK3CA PIK3R1 PTEN AKT1 MTOR TSC1 TSC2 STK11'.split(),
 'dna_repair': 'BRCA1 BRCA2 PALB2 ATM ATR CHEK2 MLH1 MSH2 MSH6 PMS2 POLE POLD1'.split(),
 'chromatin': 'ARID1A ARID1B ARID2 PBRM1 SMARCA4 BAP1 SETD2 KMT2A KMT2C KMT2D ATRX'.split(),
 'wnt': 'APC CTNNB1 AXIN1 AXIN2 RNF43'.split(),
 'metabolism': 'IDH1 IDH2 SDHB SDHC SDHD FH VHL'.split(),
}

def kind(token):
 if 'fs' in token.lower(): return 'frameshift'
 if '*' in token: return 'stop'
 if 'del' in token.lower(): return 'deletion'
 if 'ins' in token.lower(): return 'insertion'
 m = re.fullmatch(r'([A-Z])(\d+)([A-Z])', token)
 if m: return 'synonymous' if m[1] == m[3] else 'substitution'
 return 'other'

class MutationFeatures(BaseEstimator, TransformerMixin):
 def __init__(self, mode='full', min_df=3):
  self.mode = mode
  self.min_df = min_df

 def dictionaries(self, X):
  genes = list(X.columns)
  for row in X.itertuples(index=False, name=None):
   d = {}; counts = {}; burden = 0; events = 0; positions = []
   changed = set()
   for gene, value in zip(genes, row):
    if value is None or str(value) in ('nan', '', '<NA>'):
     d['missing:' + gene] = 1; continue
    if value == 'WT': continue
    changed.add(gene); burden += 1; d['gene:' + gene] = 1
    if self.mode == 'binary': continue
    tokens = str(value).split()
    for raw_token in tokens:
     token = re.sub(r'^([A-Z]\d+)X$', r'\1*', raw_token)
     events += 1; typ = kind(token); counts[typ] = counts.get(typ, 0) + 1
     d['variant:' + gene + ':' + token] = 1
     d['type:' + gene + ':' + typ] = 1
     m = re.search(r'\d+', token)
     if m:
      position = int(m[0]); positions.append(position)
      d['position:' + gene + ':' + str(position)] = 1
   if self.mode == 'full':
    d['numeric:log_mutated_genes'] = np.log1p(burden)
    d['numeric:log_events'] = np.log1p(events)
    d['numeric:multi_event_excess'] = np.log1p(max(0, events-burden))
    for typ in ['frameshift','stop','deletion','insertion','synonymous','substitution','other']:
     d['numeric:log_' + typ] = np.log1p(counts.get(typ,0))
     d['numeric:fraction_' + typ] = counts.get(typ,0)/max(events,1)
    d['numeric:log_mean_position'] = np.log1p(np.mean(positions)) if positions else 0.
    for name, members in GROUPS.items():
     d['module:' + name] = np.log1p(len(changed.intersection(members)))
   yield d

 def fit(self, X, y=None):
  self.genes_ = list(X.columns)
  self.vectorizer_ = DictVectorizer(dtype=np.float32)
  matrix = self.vectorizer_.fit_transform(self.dictionaries(X))
  names = self.vectorizer_.get_feature_names_out()
  self.keep_ = (np.asarray((matrix != 0).sum(axis=0)).ravel() >= self.min_df) | np.array([s.startswith(('numeric:', 'module:')) for s in names])
  self.feature_names_ = names[self.keep_]
  return self

 def transform(self, X):
  missing = set(self.genes_) - set(X.columns)
  if missing: raise ValueError('Missing gene columns: ' + str(sorted(missing)[:10]))
  return sparse.csr_matrix(self.vectorizer_.transform(self.dictionaries(X[self.genes_]))[:, self.keep_])

 def get_feature_names_out(self, input_features=None): return self.feature_names_
def canon(t):
 t=t.strip().removeprefix('p.');t=re.sub(r'(?<=\d)X$','*',t)
 return re.sub(r'^([A-Z]\d+)(?:[A-Z*]+)?fs.*$',r'\1fs',t)
def aa_change(t):
 m=re.fullmatch(r'([A-Z])\d+([A-Z*])',t)
 if m:return m[1]+'>'+m[2]
 m=re.match(r'([A-Z])\d+',t)
 if m:
  for suffix in ['fs','del','ins']:
   if suffix in t:return m[1]+'>'+suffix
 return None

def parse_events(frame):
 names=list(frame.columns);result=[]
 for row in frame.itertuples(index=False,name=None):
  result.append([(g,tuple(sorted(set(canon(t) for t in v.split())))) for g,v in zip(names,row) if isinstance(v,str) and v!='WT'])
 return result

class EvidenceEncoder:
 def __init__(self,unknown=1.,silent=1.,support=2):self.unknown=unknown;self.silent=silent;self.support=support
 def fit(self,events):
  self.token_df={};self.change_df={}
  if self.unknown<1:
   tokens=Counter();changes=Counter()
   for row in events:
    for g,ts in row:
     tokens.update((g,t) for t in ts)
     changes.update(set((g,a) for t in ts if (a:=aa_change(t)) is not None))
   self.token_df={k:v for k,v in tokens.items() if v>=self.support}
   self.change_df={k:v for k,v in changes.items() if v>=self.support}
  return self
 def transform(self,events,training=False,exclude_events=None):
  result=[]
  for row_number,row in enumerate(events):
   own=exclude_events[row_number] if exclude_events is not None else (row if training else [])
   own_tokens={(gg,tt) for gg,tts in own for tt in tts}
   own_changes={(gg,ac) for gg,tts in own for tt in tts if (ac:=aa_change(tt)) is not None}
   d={};gene_weights={};type_sums=Counter();non_syn=0.;trunc=0.;multikind=0
   for g,ts in row:
    types={};gene=0.;known_gene=False
    for t in ts:
     typ=kind(t);a=aa_change(t)
     known=self.token_df.get((g,t),0)-int((g,t) in own_tokens)>=self.support
     known_aa=a is not None and self.change_df.get((g,a),0)-int((g,a) in own_changes)>=self.support
     if typ=='synonymous':w=self.silent
     elif known or typ in ['stop','frameshift','deletion','insertion']:w=1.
     elif known_aa:w=max(self.unknown,.75)
     else:w=self.unknown
     if w<=0:continue
     gene=max(gene,w);types[typ]=max(types.get(typ,0),w)
     d['variant:'+g+':'+t]=max(d.get('variant:'+g+':'+t,0),w)
     m=re.search(r'\d+',t)
     if m:d['position:'+g+':'+m[0]]=max(d.get('position:'+g+':'+m[0],0),w)
     if a:d['aa:'+g+':'+a]=max(d.get('aa:'+g+':'+a,0),w)
     if known and typ!='synonymous':known_gene=True
    if gene<=0:continue
    d['gene:'+g]=gene;gene_weights[g]=gene
    for typ,w in types.items():d['type:'+g+':'+typ]=w;type_sums[typ]+=w
    if known_gene:d['supported:'+g]=1.
    non_syn+=max([w for k,w in types.items() if k!='synonymous'] or [0.])
    trunc+=max([w for k,w in types.items() if k in ['stop','frameshift','deletion','insertion']] or [0.])
    multikind+=len(types)>1
   n=len(row)
   d['numeric:log_raw_genes']=np.log1p(n)
   d['numeric:log_effective_genes']=np.log1p(sum(gene_weights.values()))
   d['numeric:log_nonsyn_genes']=np.log1p(non_syn)
   d['numeric:log_trunc_genes']=np.log1p(trunc)
   d['numeric:log_multikind_genes']=np.log1p(multikind)
   for typ in ['synonymous','substitution','stop','frameshift','deletion','insertion','other']:
    d['numeric:gene_fraction_'+typ]=type_sums[typ]/max(n,1)
   for name,gs in GROUPS.items():d['module:'+name]=np.log1p(sum(gene_weights.get(g,0) for g in gs))
   result.append(d)
  return result

events=parse_events(X)
legacy_records=list(MutationFeatures(mode='full').dictionaries(X))
assert canon('R213X')=='R213*' and canon('K16Rfs')=='K16fs'
assert aa_change('R132H')==aa_change('R140H')=='R>H'
print('Parsed rows:',len(events),'Normalization examples passed')


Parsed rows: 6201 Normalization examples passed


In [4]:
def relevance_scores(a,target,method):
 base=method.removesuffix('_full')
 if base=='embedded':
  scaled=csr32(MaxAbsScaler().fit_transform(a))
  fitted=OneVsRestClassifier(LogisticRegression(solver='liblinear',l1_ratio=1,C=.2,class_weight='balanced',max_iter=2000),n_jobs=1).fit(scaled,target)
  return np.max(np.abs(np.vstack([e.coef_[0] for e in fitted.estimators_])),axis=0)
 return (chi2(a,target) if base=='chi2' else f_classif(a,target))[0]

def csr32(a):
 b=sparse.csr_matrix(a,dtype=np.float32)
 if b.nnz>=np.iinfo(np.int32).max:raise ValueError('Sparse matrix too large for tree index type')
 b.indices=b.indices.astype(np.int32,copy=False);b.indptr=b.indptr.astype(np.int32,copy=False)
 return b

class Space:
 def fit(self,rs):
  self.vec=DictVectorizer(dtype=np.float32);a=self.vec.fit_transform(rs)
  names=self.vec.get_feature_names_out();df=np.asarray((a!=0).sum(axis=0)).ravel()
  keep=(df>=3)|np.array([n.startswith(('numeric:','module:')) for n in names])
  self.raw=np.flatnonzero(keep);self.names=names[keep]
  return csr32(a[:,self.raw])
 def transform(self,rs):return csr32(self.vec.transform(rs)[:,self.raw])
 def compact_vec(self,indices):
  v=copy.deepcopy(self.vec);mask=np.zeros(len(v.feature_names_),bool);mask[self.raw[np.sort(indices)]]=True
  v.restrict(mask);return v

class Selector:
 def __init__(self,a,target,names):
  self.a=a;self.y=target;self.names=names;self.cache={};self.corr={};self.scores={};self.audit=[];self.examples=[]
 def order(self,method,threshold):
  key=(method,threshold)
  if key in self.cache:return self.cache[key]
  if method not in self.scores:
   with warnings.catch_warnings():
    warnings.simplefilter('ignore',RuntimeWarning);score=relevance_scores(self.a,self.y,method)
   score=np.nan_to_num(score,nan=0,posinf=np.finfo(np.float32).max,neginf=0)
   if not method.endswith('_full'):score[np.array([n.startswith('numeric:') for n in self.names])]=0
   self.scores[method]=score
  score=self.scores[method]
  if method not in self.corr:
   pool=np.argsort(-score,kind='stable')[:min(4096,int((score>0).sum()))]
   z=self.a[:,pool];mean=np.asarray(z.mean(axis=0)).ravel();var=np.asarray(z.multiply(z).mean(axis=0)).ravel()-mean**2
   sd=np.sqrt(np.maximum(var,1e-12));corr=(z.T@z).toarray()/z.shape[0]-np.outer(mean,mean)
   corr/=np.outer(sd,sd);np.clip(corr,-1,1,out=corr)
   self.corr[method]=(pool,corr)
  pool,corr=self.corr[method];chosen=[]
  for i in range(len(pool)):
   if chosen:
    correlations=np.abs(corr[i,chosen]);j=int(np.argmax(correlations))
    if correlations[j]>=threshold:
     if len(self.examples)<100:self.examples.append({'method':method,'threshold':threshold,'kept':self.names[pool[chosen[j]]],'discarded':self.names[pool[i]],'correlation':float(corr[i,chosen[j]])})
     continue
   chosen.append(i)
   if len(chosen)>=2048:break
  order=pool[chosen];self.cache[key]=order
  self.audit.append({'method':method,'threshold':threshold,'association_pool':len(pool),'remaining_up_to_2048':len(order)})
  return order
 def select(self,method,threshold,k):return np.sort(self.order(method,threshold)[:k])

class CompactPredictor:
 def __init__(self,space,cols,est):
  self.vec=space.compact_vec(cols);self.names=self.vec.get_feature_names_out();self.est=est
  self.direct_inputs=len(self.names);self.raw_inputs=len(self.names);self.kind='compact'
 def predict(self,rs):return self.est.predict_proba(csr32(self.vec.transform(rs)))

def background_frame(frame,training_frame,seed,count):
 rng=np.random.default_rng(seed);out=frame.copy();arr=out.to_numpy(copy=True)
 rate=(training_frame.notna()&training_frame.ne('WT')).mean().to_numpy();prob=rate/rate.sum();idx=np.flatnonzero(prob>0)
 for r in range(len(arr)):
  n=min(len(idx),int(rng.poisson(count)))
  for c in rng.choice(len(prob),size=n,replace=False,p=prob):
   token='A1A' if rng.random()<.5 else 'A1V'
   arr[r,c]=token if not isinstance(arr[r,c],str) or arr[r,c]=='WT' else arr[r,c]+' '+token
 return pd.DataFrame(arr,columns=frame.columns,index=frame.index)
class ColumnPost:
 def __init__(self,mode='none'):self.mode=mode
 def fit(self,a,names):
  self.raw_names=np.array(names);self.names=np.array(names)
  if self.mode=='maxabs':self.scaler=MaxAbsScaler().fit(a)
  if self.mode=='block':
   self.binary=np.flatnonzero([not n.startswith(('numeric:','module:')) for n in names])
   self.summary=np.flatnonzero([n.startswith(('numeric:','module:')) for n in names])
   self.tf=TfidfTransformer(norm='l2',use_idf=True).fit(a[:,self.binary])
   self.scaler=MaxAbsScaler().fit(a[:,self.summary])
   self.names=np.concatenate([self.raw_names[self.binary],self.raw_names[self.summary]])
  return self
 def transform(self,a):
  if self.mode=='maxabs':return csr32(self.scaler.transform(a))
  if self.mode=='block':return csr32(sparse.hstack([self.tf.transform(a[:,self.binary]),self.scaler.transform(a[:,self.summary])],format='csr'))
  return csr32(a)

class Member:
 def __init__(self,name,view,encoder,space,cols,post,est):
  self.name=name;self.view=view;self.encoder=encoder;self.vec=space.compact_vec(cols);self.post=post;self.est=est
  self.raw_names=self.vec.get_feature_names_out();self.names=post.names
 def predict_rows(self,rs):return self.est.predict_proba(self.post.transform(csr32(self.vec.transform(rs))))

VIEWS={'raw':(1.,1.),'syn_down':(1.,.25),'gated':(.25,1.),'gated_syn_down':(.25,.25),'gated_nonsyn':(.25,0.)}
CONFIGS={}
for view in VIEWS:
 for k in [1024,2048]:
  for c in [.1,.5]:CONFIGS[f'{view}_lr_k{k}_C{c}']=dict(view=view,algo='lr',k=k,C=c,post='none',method='chi2_full')
  CONFIGS[f'{view}_lgb_k{k}']=dict(view=view,algo='lgb',k=k,post='none',method='f_classif_full')
for view in ['raw','gated_syn_down']:
 for depth in [3,5]:CONFIGS[f'{view}_xgb_d{depth}']=dict(view=view,algo='xgb',k=2048,depth=depth,post='none',method='f_classif_full')
 for alpha in [1,10]:CONFIGS[f'{view}_mlp_a{alpha}']=dict(view=view,algo='mlp',k=1024,alpha=alpha,post='maxabs',method='chi2_full')
 for c in [1,4]:CONFIGS[f'{view}_blocklr_C{c}']=dict(view=view,algo='lr',k=2048,C=c,post='block',method='chi2_full')
 CONFIGS[f'{view}_extra']=dict(view=view,algo='extra',k=1024,post='none',method='f_classif_full')
(OUT/'configurations.json').write_text(json.dumps(CONFIGS,indent=2))

def estimator(cfg):
 algo=cfg['algo']
 if algo=='lr':return LogisticRegression(C=cfg['C'],class_weight='balanced',max_iter=2000)
 if algo=='lgb':return LGBMClassifier(n_estimators=300,learning_rate=.04,num_leaves=15,min_child_samples=20,reg_lambda=10,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=SEED)
 if algo=='xgb':return XGBClassifier(n_estimators=300,max_depth=cfg['depth'],learning_rate=.04,min_child_weight=3,reg_lambda=10,subsample=.85,colsample_bytree=.85,max_bin=63,tree_method='hist',objective='multi:softprob',eval_metric='mlogloss',n_jobs=2,random_state=SEED)
 if algo=='mlp':return MLPClassifier(hidden_layer_sizes=(128,),activation='relu',alpha=cfg['alpha'],learning_rate_init=.001,batch_size=128,max_iter=250,tol=1e-4,n_iter_no_change=15,early_stopping=False,random_state=SEED)
 return ExtraTreesClassifier(n_estimators=250,max_depth=20,min_samples_leaf=3,max_features='sqrt',class_weight='balanced',n_jobs=2,random_state=SEED)

def prepare_view(idx,view):
 ev=[events[i] for i in idx];enc=EvidenceEncoder(*VIEWS[view]).fit(ev)
 rs=enc.transform(ev,training=True);space=Space();a=space.fit(rs);selector=Selector(a,yi[idx],space.names)
 return enc,space,a,selector

def train_new(name,idx,cache):
 cfg=CONFIGS[name];view=cfg['view']
 if view not in cache:cache[view]=prepare_view(idx,view)
 enc,space,a,selector=cache[view];cols=selector.select(cfg['method'],.98,cfg['k'])
 post=ColumnPost(cfg['post']).fit(a[:,cols],space.names[cols]);tx=post.transform(a[:,cols]);target=yi[idx]
 row_weights=np.ones(len(idx),dtype=float)
 if cfg.get('augment',False):
  if '_aug_events' not in cache:
   cache['_aug_events']=parse_events(rare_background(X.iloc[idx],[events[i] for i in idx],SEED+777,count=10))
  aug_rs=enc.transform(cache['_aug_events'],exclude_events=[events[i] for i in idx])
  aug_x=post.transform(space.transform(aug_rs)[:,cols]);tx=csr32(sparse.vstack([tx,aug_x],format='csr'))
  target=np.concatenate([target,target]);row_weights=np.concatenate([row_weights,.5*row_weights])
 est=estimator(cfg)
 if cfg['algo'] in ['xgb','mlp']:est.fit(tx,target,sample_weight=compute_sample_weight('balanced',target)*row_weights)
 elif cfg.get('augment',False):est.fit(tx,target,sample_weight=row_weights)
 else:est.fit(tx,target)
 return Member(name,view,enc,space,cols,post,est)

LEGACY={
 'legacy_lr':dict(k=2048,method='chi2',algo='lr'),
 'legacy_lgb':dict(k=2048,method='f_classif_full',algo='lgb'),
 'legacy_extra':dict(k=1024,method='f_classif_full',algo='extra')}
def train_legacy(idx):
 space=Space();a=space.fit([legacy_records[i] for i in idx]);sel=Selector(a,yi[idx],space.names);result=[]
 for name,cfg in LEGACY.items():
  cols=sel.select(cfg['method'],.95,cfg['k'])
  if cfg['algo']=='lr':est=LogisticRegression(C=.1,class_weight='balanced',max_iter=2000)
  elif cfg['algo']=='lgb':est=LGBMClassifier(n_estimators=250,learning_rate=.05,num_leaves=15,min_child_samples=20,reg_lambda=10,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=SEED)
  else:est=ExtraTreesClassifier(n_estimators=250,max_depth=20,min_samples_leaf=3,max_features='sqrt',class_weight='balanced',n_jobs=2,random_state=SEED)
  est.fit(a[:,cols],yi[idx]);post=ColumnPost().fit(a[:,cols],space.names[cols]);result.append(Member(name,'legacy',None,space,cols,post,est))
 return result

def duplicate_frame(frame):
 arr=frame.to_numpy(copy=True)
 for i,row in enumerate(arr):
  for j,v in enumerate(row):
   if isinstance(v,str) and v!='WT':arr[i,j]=v+' '+v
 return pd.DataFrame(arr,columns=frame.columns,index=frame.index)

def rare_background(frame,tr_events,seed,count=20):
 rng=np.random.default_rng(seed);df=Counter((g,t) for row in tr_events for g,ts in row for t in ts)
 syn=[];sub=[]
 for (g,t),n in df.items():
  if n>2:continue
  if kind(t)=='synonymous':syn.append((g,t))
  elif kind(t)=='substitution':sub.append((g,t))
 arr=frame.to_numpy(copy=True);lookup={g:i for i,g in enumerate(frame.columns)}
 for row in arr:
  for _ in range(int(rng.poisson(count))):
   pool=syn if rng.random()<.5 else sub
   g,t=pool[int(rng.integers(len(pool)))];j=lookup[g]
   row[j]=t if not isinstance(row[j],str) or row[j]=='WT' else row[j]+' '+t
 return pd.DataFrame(arr,columns=frame.columns,index=frame.index)

def eval_views(idx,tr_idx,seed):
 clean=X.iloc[idx];noise=rare_background(clean,[events[i] for i in tr_idx],seed);dup=duplicate_frame(clean)
 frames=[clean,noise,dup]
 return frames,[parse_events(f) for f in frames],[list(MutationFeatures(mode='full').dictionaries(f)) for f in frames]

def predict_members(members,weights,frame):
 ev=None;old=None;cache={};p=None
 for m,w in zip(members,weights):
  if m.view=='legacy':
   if old is None:old=list(MutationFeatures(mode='full').dictionaries(frame[genes]))
   rs=old
  else:
   if ev is None:ev=parse_events(frame[genes])
   if m.view not in cache:cache[m.view]=m.encoder.transform(ev)
   rs=cache[m.view]
  q=m.predict_rows(rs);p=q*w if p is None else p+q*w
 return p
print('Historical definitions loaded; no historical search rerun.')

# CV-guided targeted refinement: regularization, support threshold, and rare-variant augmentation.
VIEWS['gated_syn_down_s3']=(.25,.25,3)
CONFIGS.update({'raw_mlp_a0.01': {'view': 'raw', 'algo': 'mlp', 'k': 1024, 'alpha': 0.01, 'post': 'maxabs', 'method': 'chi2_full'}, 'raw_mlp_a0.1': {'view': 'raw', 'algo': 'mlp', 'k': 1024, 'alpha': 0.1, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_mlp_a0.01': {'view': 'gated_syn_down', 'algo': 'mlp', 'k': 1024, 'alpha': 0.01, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_mlp_a0.1': {'view': 'gated_syn_down', 'algo': 'mlp', 'k': 1024, 'alpha': 0.1, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k1024_C0.1': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 1024, 'C': 0.1, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k1024_C0.5': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 1024, 'C': 0.5, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lgb_k1024': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 1024, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_lr_k2048_C0.1': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 2048, 'C': 0.1, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k2048_C0.5': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 2048, 'C': 0.5, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lgb_k2048': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_xgb_d3': {'view': 'gated_syn_down_s3', 'algo': 'xgb', 'k': 2048, 'depth': 3, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_xgb_d5': {'view': 'gated_syn_down_s3', 'algo': 'xgb', 'k': 2048, 'depth': 5, 'post': 'none', 'method': 'f_classif_full'}, 'raw_aug_lgb': {'view': 'raw', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full', 'augment': True}, 'raw_aug_xgb_d5': {'view': 'raw', 'algo': 'xgb', 'k': 2048, 'depth': 5, 'post': 'none', 'method': 'f_classif_full', 'augment': True}, 'gated_s3_aug_lgb': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full', 'augment': True}})
(OUT/'configurations.json').write_text(json.dumps(CONFIGS,indent=2))
print('Historical comparator definitions ready.')


Novel settings: 44 Views: {'raw': (1.0, 1.0), 'syn_down': (1.0, 0.25), 'gated': (0.25, 1.0), 'gated_syn_down': (0.25, 0.25), 'gated_nonsyn': (0.25, 0.0)}
Total new configurations after refinement: 59


In [5]:
# Freeze the previous version; do not repeat its 59-setting search.
BASE_CONFIGS=copy.deepcopy(CONFIGS)
BASE_PLAN=[('v4_reference',.25),('gated_s3_aug_lgb',.1875),('raw_aug_xgb_d5',.1875),('raw_lr_k2048_C0.1',.1875),('gated_syn_down_mlp_a1',.1875)]

def fit_base(idx):
 cache={};ms=[];ws=[]
 for name,w in BASE_PLAN:
  if name=='v4_reference':ms.extend(train_legacy(idx));ws.extend([w/3]*3)
  else:ms.append(train_new(name,idx,cache));ws.append(w)
 return ms,np.array(ws)


In [6]:
class NewSelector:
 def __init__(self,a,target,names):self.a=a;self.y=target;self.names=names;self.cache={}
 def select(self,method,k):
  if (method,k) in self.cache:return self.cache[(method,k)]
  numeric=np.array([s.startswith(('numeric:','module:')) for s in self.names])
  with warnings.catch_warnings():
   warnings.simplefilter('ignore',RuntimeWarning)
   global_score=np.nan_to_num(f_classif(self.a,self.y)[0],nan=0,posinf=0,neginf=0)
   if method=='balanced':
    orders=[]
    for c in range(NC):
     yes=self.y==c
     score=np.nan_to_num(chi2(self.a,yes)[0],nan=0,posinf=0,neginf=0)
     high=np.asarray(self.a[yes].mean(axis=0)).ravel()>np.asarray(self.a[~yes].mean(axis=0)).ravel()
     score[~high|numeric]=0
     orders.append(np.argsort(-score,kind='stable')[:min(k,int((score>0).sum()))])
    order=[]
    for rank in range(max(map(len,orders))):
     for o in orders:
      if rank<len(o):order.append(int(o[rank]))
    order+=list(np.argsort(-global_score,kind='stable'))
   else:order=list(np.argsort(-global_score,kind='stable'))
  z=self.a.tocsc();chosen=list(np.flatnonzero(numeric));seen=set(chosen);signatures=set()
  for j in order:
   j=int(j)
   if j in seen:continue
   seen.add(j)
   if global_score[j]<=0:continue
   lo,hi=z.indptr[j:j+2]
   sig=hashlib.sha256(z.indices[lo:hi].tobytes()+z.data[lo:hi].tobytes()).digest()
   if sig in signatures:continue
   signatures.add(sig);chosen.append(j)
   if len(chosen)>=k:break
  self.cache[(method,k)]=np.sort(chosen);return self.cache[(method,k)]

class SpecificityPost:
 def fit(self,a,names,target):
  self.names=np.array(names);self.binary=np.flatnonzero([not n.startswith(('numeric:','module:')) for n in names]);self.summary=np.flatnonzero([n.startswith(('numeric:','module:')) for n in names])
  z=(a[:,self.binary]!=0).astype(np.float32)
  counts=np.vstack([np.asarray(z[target==c].sum(axis=0)).ravel() for c in range(NC)])
  prior=np.bincount(target,minlength=NC)/len(target);freq=counts.sum(axis=0)
  posterior=(counts+10*prior[:,None])/(freq+10)
  kl=(posterior*np.log(np.maximum(posterior,1e-12)/prior[:,None])).sum(axis=0)
  self.weights=(.25+np.sqrt(np.maximum(kl,0))*freq/(freq+20)).astype(np.float32)
  self.scaler=MaxAbsScaler().fit(a[:,self.summary]);return self
 def transform(self,a):
  from sklearn.preprocessing import normalize
  block=normalize(a[:,self.binary].multiply(self.weights),norm='l2')
  return csr32(sparse.hstack([block,self.scaler.transform(a[:,self.summary])],format='csr'))

NEW={}
for view in ['raw','gated_syn_down_s3']:
 for method in ['global','balanced']:
  NEW[f'{view}_{method}_lgb4096']=dict(view=view,algo='lgb',k=4096,method=method)
for c in [.1,.5]:NEW[f'balanced_lr_C{c}']=dict(view='raw',algo='lr',k=2048,method='balanced',C=c)
for c in [1.,4.]:NEW[f'specificity_lr_C{c}']=dict(view='raw',algo='lr',k=4096,method='balanced',C=c,specificity=True)

def fit_new6(name,idx,cache):
 cfg=NEW[name];view=cfg['view']
 if view not in cache:
  enc,space,a,_=prepare_view(idx,view);cache[view]=(enc,space,a,NewSelector(a,yi[idx],space.names))
 enc,space,a,sel=cache[view];cols=sel.select(cfg['method'],cfg['k']);target=yi[idx]
 if cfg.get('specificity'):post=SpecificityPost().fit(a[:,cols],space.names[cols],target)
 else:post=ColumnPost().fit(a[:,cols],space.names[cols])
 est=estimator(cfg);tx=post.transform(a[:,cols])
 if cfg.get('augment'):
  if '_aug' not in cache:cache['_aug']=parse_events(rare_background(X.iloc[idx],[events[i] for i in idx],SEED+777,count=10))
  ar=enc.transform(cache['_aug'],exclude_events=[events[i] for i in idx]);ax=post.transform(space.transform(ar)[:,cols])
  tx=csr32(sparse.vstack([tx,ax],format='csr'));target=np.concatenate([target,target]);rw=np.r_[np.ones(len(idx)),np.full(len(idx),.5)]
 else:rw=np.ones(len(idx))
 if cfg['algo']=='xgb':est.fit(tx,target,sample_weight=compute_sample_weight('balanced',target)*rw)
 elif cfg.get('augment'):est.fit(tx,target,sample_weight=rw)
 else:est.fit(tx,target)
 m=Member(name,view,enc,space,cols,post,est)
 # Column order of a specificity block changes, but raw_names always describe the vectorizer input.
 if cfg.get('specificity'):m.names=np.concatenate([m.raw_names[post.binary],m.raw_names[post.summary]])
 return m
(OUT/'new_configurations.json').write_text(json.dumps(NEW,indent=2))
print('Frozen v6 component definitions loaded.')

NEW.update({'gated_s3_aug_global_lgb4096': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 4096, 'method': 'global', 'augment': True}, 'gated_s3_aug_balanced_lgb4096': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 4096, 'method': 'balanced', 'augment': True}, 'raw_aug_global_xgb4096_d3': {'view': 'raw', 'algo': 'xgb', 'k': 4096, 'method': 'global', 'depth': 3, 'augment': True}, 'raw_aug_global_xgb4096_d5': {'view': 'raw', 'algo': 'xgb', 'k': 4096, 'method': 'global', 'depth': 5, 'augment': True}})
(OUT/'new_configurations.json').write_text(json.dumps(NEW,indent=2))

print('Frozen v6 comparator ready.')


New configurations 8
Total new configurations including augmentation extension: 12


In [7]:
def fit_plan6(idx,plan,basefit=None):
 ms=[];ws=[];cache={}
 for name,w in plan:
  if name=='v5_reference':
   old,ow=basefit if basefit is not None else fit_base(idx)
   ms.extend(old);ws.extend(ow*w)
  else:ms.append(fit_new6(name,idx,cache));ws.append(w)
 return ms,np.array(ws)

V6_PLAN=[('v5_reference',.75),('gated_s3_aug_global_lgb4096',.25)]


## 1. 고정 구성과 비교 계획
v6의 원래 가중치는 legacy 3개 각각 0.0625, v5 신규 4개 각각 0.140625, 넓은 LGB 0.25입니다. 한 모델의 상대 가중치를 0/0.5/1.5/2배로 바꾸고 합이 1이 되게 정규화합니다. 전체/신규 균등, legacy 제거, 트리/비트리 그룹 등을 사전에 정한 제한된 계획으로 비교합니다. 중복된 가중치 계획은 제거합니다.

모델 자체의 정확도가 낮다는 이유만으로 제거하지 않습니다. 모델을 뺐을 때 앙상블 F1 변화와 다른 모델과의 오류 중복을 함께 기록합니다. 재분할 결과로 새 가중치를 계속 조정하지 않습니다.

In [8]:
NAMES9=['legacy_lr','legacy_lgb','legacy_extra','gated_s3_aug_lgb','raw_aug_xgb_d5','raw_lr_k2048_C0.1','gated_syn_down_mlp_a1','gated_s3_aug_global_lgb4096']
W6=np.array([.0625,.0625,.0625,.140625,.140625,.140625,.140625,.25],dtype=float)
TREE9=np.array([False,True,True,True,True,False,False,True])
WIDTH9=np.array([2048,2048,1024,2048,2048,2048,1024,4096])
CACHE_COMPONENTS=True
CONFIRM_WORKERS=2
CONFIRM_SEEDS=[109,211]

def fit_components9(idx):
 ms,_=fit_plan6(idx,V6_PLAN);byname={m.name:m for m in ms}
 assert set(byname)==set(NAMES9)
 return [byname[name] for name in NAMES9]

def component_probabilities9(ms,frames,evs=None,legacy=None):
 if evs is None:evs=[parse_events(f[genes]) for f in frames]
 if legacy is None:legacy=[list(MutationFeatures(mode='full').dictionaries(f[genes])) for f in frames]
 out=np.empty((len(ms),len(frames),len(frames[0]),NC),np.float32);cache={}
 for i,m in enumerate(ms):
  for j,frame in enumerate(frames):
   if m.view=='legacy':rs=legacy[j]
   else:
    key=(m.view,j)
    if key not in cache:cache[key]=m.encoder.transform(evs[j])
    rs=cache[key]
   out[i,j]=m.predict_rows(rs)
 return out

def weighted9(prob,w):return np.tensordot(np.asarray(w,dtype=np.float64),prob,axes=(0,0))
def scores9(prob,splits):
 fs=np.array([[f1_score(yi[dev[b]],prob[j,b].argmax(axis=1),average='macro',zero_division=0) for j in range(3)] for _,b in splits])
 return dict(clean=fs[:,0].mean(),noise=fs[:,1].mean(),duplicate=fs[:,2].mean(),score=(fs@np.array([.6,.3,.1])).mean(),clean_std=fs[:,0].std(ddof=1))

PLANS9={};seen9=set()
def add_plan9(name,w):
 w=np.asarray(w,float)
 if w.sum()<=0:return
 w=w/w.sum();signature=tuple(np.round(w,12))
 if signature in seen9:return
 seen9.add(signature);PLANS9[name]=w
add_plan9('v6_reference',W6)
for i,name in enumerate(NAMES9):
 for mult in [0.,.5,1.5,2.]:
  w=W6.copy();w[i]*=mult;add_plan9(f'{name}_relative_{mult}',w)
add_plan9('equal_all',np.ones(8));add_plan9('equal_nonlegacy',np.r_[np.zeros(3),np.ones(5)])
w=W6.copy();w[:3]=0;add_plan9('drop_all_legacy',w)
for scale in [.75,1.25]:
 w=W6.copy();w[TREE9]*=scale;add_plan9(f'tree_group_relative_{scale}',w)
add_plan9('tree_only',W6*TREE9);add_plan9('non_tree_only',W6*(~TREE9))
(OUT/'weight_plans.json').write_text(json.dumps({k:v.tolist() for k,v in PLANS9.items()},indent=2))
print('Fixed unique weight plans:',len(PLANS9),'base classifiers:',len(NAMES9))

# After the initial 40 one-at-a-time/group comparisons, test a bounded joint-weight grid.
rng9=np.random.default_rng(904)
for concentration in [24.,64.]:
 for trial in range(32):
  w=rng9.dirichlet(W6*concentration)
  if trial%2:
   w[w<.03]=0
  add_plan9(f'joint_dirichlet_{concentration}_{trial:02d}',w)
(OUT/'weight_plans.json').write_text(json.dumps({k:v.tolist() for k,v in PLANS9.items()},indent=2))
print('Total fixed plans after bounded joint extension:',len(PLANS9))


Fixed unique weight plans: 40 base classifiers: 8


Bounded joint-weight extension: 64 additional plans, total 104.


## 2. 같은 분할의 구성별 OOF 생성과 캐시 검증
v5의 신규 4개·v6의 넓은 LGB의 완료 OOF는 데이터 해시·클래스·분할·원본 프로필을 확인한 경우 재사용합니다. legacy 3개는 실제로 매 fold 재학습하고 모든 조건의 확률을 생성합니다. 이 8개 확률을 원래 v6 가중치로 합한 값이 완료된 v6 전체 OOF와 일치해야 캐시를 인정합니다. 전체 train으로 학습한 모델은 이 단계에 사용하지 않습니다. 캐시 파일이 없으면 전부 학습합니다.

In [9]:
P9=np.zeros((8,3,len(dev),NC),np.float32);cached9=False;cache_models9=[]
v5_dir=WORK/'artifacts'/'v5';v6_dir=WORK/'artifacts'/'v6'
if CACHE_COMPONENTS and all(p.exists() for p in [v5_dir/'oof_probabilities.npz',v5_dir/'verification.json',v6_dir/'oof_probabilities.npz',v6_dir/'split_manifest.csv']):
 proof5=json.loads((v5_dir/'verification.json').read_text())
 for fname in ['train.csv','test.csv']:
  assert hashlib.sha256((DATA/fname).read_bytes()).hexdigest()==proof5['sha256'][fname], 'Cached dataset differs'
 saved=pd.read_csv(v6_dir/'split_manifest.csv',dtype={'group':str});assert saved.ID.equals(train.ID) and np.array_equal(saved.group.to_numpy(),groups.astype(str))
 q5=np.load(v5_dir/'oof_probabilities.npz',allow_pickle=True);q6=np.load(v6_dir/'oof_probabilities.npz',allow_pickle=True)
 assert np.array_equal(q5['dev'],dev) and np.array_equal(q6['dev'],dev) and np.array_equal(q5['labels'],labels) and np.array_equal(q6['labels'],labels)
 for i in [3,4,5,6]:P9[i]=q5[NAMES9[i]];cache_models9.append(NAMES9[i])
 P9[7]=q6[NAMES9[7]];cache_models9.append(NAMES9[7]);cached9=True
 baseline_cache9=.75*q6['v5_reference']+.25*q6['gated_s3_aug_global_lgb4096']
for fold,(a,b) in enumerate(folds):
 tr=dev[a];va=dev[b];frames,evs,olds=eval_views(va,tr,SEED+901+fold)
 ms=train_legacy(tr) if cached9 else fit_components9(tr)
 ps=component_probabilities9(ms,frames,evs,olds)
 for i,m in enumerate(ms):P9[NAMES9.index(m.name)][:,b]=ps[i]
 del ms,ps
 print('Completed component OOF fold',fold,'refitted classifiers:',3 if cached9 else 8,flush=True)
 np.savez_compressed(OUT/'component_oof.npz',probabilities=P9,dev=dev,labels=labels,names=np.array(NAMES9))
assert np.isfinite(P9).all() and np.allclose(P9.sum(axis=3),1)
reference9=weighted9(P9,W6)
if cached9:assert np.allclose(reference9,baseline_cache9,atol=1e-6),'Reconstructed baseline differs from v6'
rows9=[]
for i,name in enumerate(NAMES9):rows9.append(dict(model=name,**scores9(P9[i],folds),inputs=int(WIDTH9[i])))
component_scores9=pd.DataFrame(rows9);component_scores9.to_csv(OUT/'component_scores.csv',index=False)
rank9=pd.DataFrame({name:dict(**scores9(weighted9(P9,w),folds),models=int((w>0).sum()),inputs=int(WIDTH9[w>0].sum())) for name,w in PLANS9.items()}).T.sort_values('score',ascending=False)
rank9.to_csv(OUT/'weight_comparison.csv');display(rank9.head(15));display(component_scores9)
# Paired error diagnostics use development OOF, not test labels.
errors9=[];pred9=P9[:,0].argmax(axis=2);target=yi[dev]
for i in range(8):
 for j in range(i+1,8):errors9.append(dict(first=NAMES9[i],second=NAMES9[j],agreement=float(np.mean(pred9[i]==pred9[j])),both_wrong=float(np.mean((pred9[i]!=target)&(pred9[j]!=target))),only_first_correct=float(np.mean((pred9[i]==target)&(pred9[j]!=target))),only_second_correct=float(np.mean((pred9[j]==target)&(pred9[i]!=target)))))
pd.DataFrame(errors9).to_csv(OUT/'error_overlap.csv',index=False)
ablation9=rank9.loc[[f'{name}_relative_0.0' for name in NAMES9]].copy()
for col in ['clean','noise','duplicate','score']:ablation9['delta_'+col]=ablation9[col]-rank9.loc['v6_reference',col]
ablation9.to_csv(OUT/'leave_one_out_comparison.csv');display(ablation9)
# Freeze at most two finalists with different predeclared objectives.
base=rank9.loc['v6_reference'];FINALISTS9=[];OBJECTIVES9={}
clean_options=rank9[(rank9.index!='v6_reference')&(rank9.clean>=base.clean+.003)&(rank9.noise>=base.noise-.003)&(rank9.score>=base.score)]
if len(clean_options):
 name=clean_options.sort_values(['clean','models'],ascending=[False,True]).index[0];FINALISTS9.append(name);OBJECTIVES9[name]='clean'
robust_options=rank9[(rank9.index!='v6_reference')&(rank9.score>=base.score+.002)&(rank9.clean>=base.clean-.002)]
if len(robust_options):
 best=float(robust_options.score.max());short=robust_options[robust_options.score>=best-.0005].sort_values(['models','score'],ascending=[True,False]);name=short.index[0]
 if name not in FINALISTS9:FINALISTS9.append(name);OBJECTIVES9[name]='robust'
(OUT/'selection_primary.json').write_text(json.dumps(dict(finalists=FINALISTS9,objectives=OBJECTIVES9,plans={name:PLANS9[name].tolist() for name in FINALISTS9}),indent=2))
print('Frozen finalists:',FINALISTS9,OBJECTIVES9,flush=True)


Reusing verified completed component OOF; evaluating all 104 weight plans.
Frozen finalists: [] {}


,clean,noise,duplicate,score,clean_std,models,inputs
joint_dirichlet_64.0_06,0.434316,0.353820,0.434130,0.410148,0.008220,8.0,16384.0
tree_group_relative_0.75,0.436236,0.349667,0.433553,0.409997,0.007285,8.0,16384.0
joint_dirichlet_64.0_30,0.433929,0.354318,0.432753,0.409928,0.008686,8.0,16384.0
joint_dirichlet_64.0_05,0.436907,0.347289,0.435108,0.409842,0.008721,8.0,16384.0
legacy_extra_relative_2.0,0.437323,0.345946,0.435892,0.409767,0.007690,8.0,16384.0
joint_dirichlet_64.0_12,0.434181,0.352933,0.433164,0.409705,0.007461,8.0,16384.0
legacy_lr_relative_2.0,0.435604,0.349761,0.433628,0.409653,0.009074,8.0,16384.0
joint_dirichlet_24.0_13,0.435164,0.349788,0.436154,0.409650,0.009159,8.0,16384.0
joint_dirichlet_64.0_25,0.436880,0.345785,0.436428,0.409506,0.007218,8.0,16384.0
raw_aug_xgb_d5_relative_0.5,0.436302,0.347863,0.433616,0.409502,0.008333,8.0,16384.0


,model,clean,noise,duplicate,score,clean_std,inputs
0,legacy_lr,0.374907,0.326225,0.374907,0.360302,0.011287,2048
1,legacy_lgb,0.401555,0.207571,0.352142,0.338419,0.012083,2048
2,legacy_extra,0.352212,0.279005,0.346688,0.329697,0.006214,1024
3,gated_s3_aug_lgb,0.414637,0.337619,0.414637,0.391532,0.008868,2048
4,raw_aug_xgb_d5,0.404134,0.304266,0.404134,0.374174,0.007386,2048
5,raw_lr_k2048_C0.1,0.393797,0.312149,0.393797,0.369302,0.011315,2048
6,gated_syn_down_mlp_a1,0.366081,0.304288,0.366081,0.347543,0.014446,1024
7,gated_s3_aug_global_lgb4096,0.426273,0.339723,0.426273,0.400308,0.006798,4096


,clean,noise,duplicate,score,clean_std,models,inputs,delta_clean,delta_noise,delta_duplicate,delta_score
legacy_lr_relative_0.0,0.436202,0.344681,0.433926,0.408518,0.009818,7.0,14336.0,0.000625,-0.001093,0.000624,0.000110
legacy_lgb_relative_0.0,0.432950,0.353276,0.433422,0.409095,0.008749,7.0,14336.0,-0.002626,0.007501,0.000121,0.000687
legacy_extra_relative_0.0,0.436193,0.344447,0.433024,0.408352,0.009348,7.0,15360.0,0.000616,-0.001328,-0.000277,-0.000056
gated_s3_aug_lgb_relative_0.0,0.434234,0.343672,0.434071,0.407049,0.006089,7.0,14336.0,-0.001342,-0.002102,0.000769,-0.001359
raw_aug_xgb_d5_relative_0.0,0.433364,0.349703,0.434978,0.408427,0.009220,7.0,14336.0,-0.002212,0.003928,0.001676,0.000019
raw_lr_k2048_C0.1_relative_0.0,0.433186,0.343461,0.432013,0.406151,0.009999,7.0,14336.0,-0.002391,-0.002314,-0.001289,-0.002258
gated_syn_down_mlp_a1_relative_0.0,0.433025,0.345273,0.430949,0.406492,0.009583,7.0,15360.0,-0.002551,-0.000502,-0.002352,-0.001916
gated_s3_aug_global_lgb4096_relative_0.0,0.432215,0.339619,0.433093,0.404524,0.002071,7.0,12288.0,-0.003362,-0.006155,-0.000209,-0.003885


## 3. 반복 그룹 검증: 고정 후보만 seed=109/211에서 확인
개발 CV에서 원본 F1 +0.003·스트레스 하락 0.003 이하·종합 점수 비하락인 clean 후보, 또는 종합 +0.002·원본 하락 0.002 이하인 robust 후보를 최대 2개 고정합니다. 이후 가중치를 추가로 튜닝하지 않습니다.

두 새 분할에서 v6의 8개 구성 모델을 모두 다시 학습해 동일 확률로 후보와 기준을 비교합니다. 학습 파라미터/선별은 그대로이고 확률 가중치만 다릅니다. 작업별 전처리·학습 객체는 공유하지 않으며 병렬 작업 수는 CONFIRM_WORKERS로 조절할 수 있습니다. 이 분할도 같은 개발 환자라 독립 새 데이터는 아닙니다.

각 seed의 평균 clean 하락 0.002 이하·스트레스 하락 0.005 이하를 요구합니다. 두 seed 평균에서 clean 목적 후보는 clean +0.0015 이상·종합 비하락, robust 목적 후보는 clean +0.0005 이상·종합 +0.001 이상이어야 합니다. clean 후보 우선이라는 순서는 개발 CV에서 고정하며 재분할의 최고값으로 가중치를 새로 만들지 않습니다.

In [10]:
from concurrent.futures import ThreadPoolExecutor,as_completed
confirmation_rows9=[];confirmation_probs9={seed:np.zeros_like(P9) for seed in CONFIRM_SEEDS};confirmation_splits9={}

def confirmation_task9(seed,fold,a,b):
 tr=dev[a];va=dev[b];start=time.time();ms=fit_components9(tr)
 frames,evs,olds=eval_views(va,tr,SEED+seed*10+fold)
 probability=component_probabilities9(ms,frames,evs,olds);del ms
 result=[]
 for name in ['v6_reference',*FINALISTS9]:
  p=weighted9(probability,PLANS9[name]);fs=[f1_score(yi[va],p[j].argmax(axis=1),average='macro',zero_division=0) for j in range(3)]
  result.append(dict(seed=seed,fold=fold,model=name,clean=fs[0],noise=fs[1],duplicate=fs[2],score=.6*fs[0]+.3*fs[1]+.1*fs[2]))
 return seed,fold,b,probability,result,time.time()-start

if FINALISTS9:
 tasks=[]
 for seed in CONFIRM_SEEDS:
  splits=list(StratifiedGroupKFold(3,shuffle=True,random_state=seed).split(X.iloc[dev],y[dev],groups[dev]));confirmation_splits9[seed]=splits
  for fold,(a,b) in enumerate(splits):
   assert not set(groups[dev[a]]).intersection(groups[dev[b]])
   tasks.append((seed,fold,a,b))
 with ThreadPoolExecutor(max_workers=CONFIRM_WORKERS) as pool:
  futures=[pool.submit(confirmation_task9,*task) for task in tasks]
  for future in as_completed(futures):
   seed,fold,b,probability,rows,seconds=future.result();confirmation_probs9[seed][:,:,b]=probability;confirmation_rows9.extend(rows)
   pd.DataFrame(confirmation_rows9).sort_values(['seed','fold','model']).to_csv(OUT/'confirmation_cv.csv',index=False)
   np.savez_compressed(OUT/'confirmation_component_probabilities.npz',dev=dev,labels=labels,names=np.array(NAMES9),**{f'seed_{key}':value for key,value in confirmation_probs9.items()})
   print('Confirmation completed',seed,'fold',fold,'seconds',round(seconds,1),rows,flush=True)
 confirmation9=pd.DataFrame(confirmation_rows9).groupby(['seed','model'])[['clean','noise','duplicate','score']].mean()
 display(confirmation9)
else:confirmation9=pd.DataFrame();print('No finalist passed primary rules; retain v6.')
passes9={};deltas9={}
for name in FINALISTS9:
 delta=pd.DataFrame([confirmation9.loc[(seed,name)]-confirmation9.loc[(seed,'v6_reference')] for seed in CONFIRM_SEEDS],index=CONFIRM_SEEDS)
 bounded=bool((delta.clean>=-.002).all() and (delta.noise>=-.005).all())
 average=delta.mean();goal=OBJECTIVES9[name]
 gain=bool(average.clean>=.0015 and average.score>=0) if goal=='clean' else bool(average.clean>=.0005 and average.score>=.001)
 passes9[name]=bool(bounded and gain);deltas9[name]=dict(per_seed=delta.to_dict(orient='index'),mean=average.to_dict())
winner9=next((name for name in FINALISTS9 if passes9[name]),'v6_reference');accepted9=winner9!='v6_reference';weights9=PLANS9[winner9]
(OUT/'selection.json').write_text(json.dumps(dict(winner=winner9,accepted=accepted9,weights=weights9.tolist(),objectives=OBJECTIVES9,passes=passes9,deltas=deltas9),indent=2))
print('Final winner:',winner9,'adopted:',accepted9,flush=True)


No finalist passed primary rules; retain v6.
Final winner: v6_reference adopted: False


## 4. 이전 홀드아웃 재평가·최종 모델·CSV
선택은 이미 고정했습니다. 이전 홀드아웃은 여러 번 관측한 참고값이며 후보를 다시 고르는 데 사용하지 않습니다. 전체 train 모델 캐시는 이 단계에서만 재사용하며 CV/재분할/홀드아웃에는 사용하지 않습니다. 미채택이면 기존 v6를 유지하고 중복 CSV를 새 제출로 배포하지 않습니다.

In [11]:
hold_ms9=fit_components9(dev);hold_frames=[X.iloc[hold]]
hold_components9=component_probabilities9(hold_ms9,hold_frames)[:,0]
hold_baseline9=weighted9(hold_components9,W6);hold_candidate9=weighted9(hold_components9,weights9)
metrics9={}
for title,probs in [('v6_reference',hold_baseline9),('v9_final',hold_candidate9)]:
 pred=labels[probs.argmax(axis=1)];metrics9[title]=dict(macro_f1=f1_score(y[hold],pred,average='macro',zero_division=0),accuracy=accuracy_score(y[hold],pred),weighted_f1=f1_score(y[hold],pred,average='weighted',zero_division=0),n=len(hold))
np.savez_compressed(OUT/'holdout_component_probabilities.npz',probabilities=hold_components9,indices=hold,names=np.array(NAMES9))
(OUT/'holdout_metrics.json').write_text(json.dumps(metrics9,indent=2));print('Previously observed holdout:',metrics9,flush=True);del hold_ms9
full_fit_reused9=False;full_cache_sha9=None
if CACHE_COMPONENTS and (v6_dir/'cancer_model_v6.pkl').exists() and cached9:
 with open(v6_dir/'cancer_model_v6.pkl','rb') as f:old_full=pickle.load(f)
 assert old_full['version']==6 and old_full['genes']==genes and np.array_equal(old_full['classes'],labels)
 byname={m.name:m for m in old_full['models']};assert set(byname)==set(NAMES9)
 full_ms9=[byname[name] for name in NAMES9];full_fit_reused9=True;full_cache_sha9=hashlib.sha256((v6_dir/'cancer_model_v6.pkl').read_bytes()).hexdigest()
 print('Reusing completed full-train component fits after weight selection.',flush=True)
else:full_ms9=fit_components9(np.arange(len(train)))
selected_models9=[m for m,w in zip(full_ms9,weights9) if w>0];selected_weights9=weights9[weights9>0]
feature_rows=[];sizes9=[]
for m,w in zip(selected_models9,selected_weights9):
 sizes9.append(dict(model=m.name,view=m.view,weight=float(w),inputs=len(m.names),unknown_weight=m.encoder.unknown if m.encoder else None,silent_weight=m.encoder.silent if m.encoder else None,support=m.encoder.support if m.encoder else None))
 for feature in m.raw_names:feature_rows.append(dict(model=m.name,view=m.view,feature=feature))
pd.DataFrame(feature_rows).to_csv(OUT/'final_features.csv',index=False);(OUT/'model_sizes.json').write_text(json.dumps(sizes9,indent=2))
def predict_bundle9(bundle,frame):return predict_members(bundle['models'],bundle['weights'],frame)
bundle9=dict(version=9,models=selected_models9,weights=selected_weights9,genes=genes,classes=labels,winner=winner9,holdout_metrics=metrics9['v9_final'],predict=predict_bundle9,sizes=sizes9)
model_path9=OUT/'cancer_model_v9.pkl'
with model_path9.open('wb') as f:cloudpickle.dump(bundle9,f,protocol=pickle.HIGHEST_PROTOCOL)
with model_path9.open('rb') as f:loaded9=pickle.load(f)
test_probability9=loaded9['predict'](loaded9,test)
assert np.isfinite(test_probability9).all() and np.allclose(test_probability9.sum(axis=1),1)
submission9=sample.copy();submission9.SUBCLASS=labels[test_probability9.argmax(axis=1)];assert submission9.ID.equals(test.ID)
submission9.to_csv(OUT/'submission_v9.csv',index=False)
if accepted9:submission9.to_csv(SUBMIT/'혜진_submission_v9.csv',index=False)
pd.DataFrame(test_probability9,index=test.ID,columns=labels).to_csv(OUT/'test_probabilities.csv')
print('Saved',model_path9,'MiB',round(model_path9.stat().st_size/1024**2,2),'new submission:',accepted9,flush=True)


Previously observed holdout: {'v6_reference': {'macro_f1': 0.4729888723709301, 'accuracy': 0.46333601933924257, 'weighted_f1': 0.46777775879200023, 'n': 1241}, 'v9_final': {'macro_f1': 0.4729888723709301, 'accuracy': 0.46333601933924257, 'weighted_f1': 0.46777775879200023, 'n': 1241}}
Reusing completed full-train component fits after weight selection.
Saved /Users/mac/project/cancer_project/artifacts/v9/cancer_model_v9.pkl MiB 83.09 new submission: False


/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)


## 5. 독립 로딩·문서·변수 사전

In [12]:
import subprocess,tempfile,importlib.metadata
program="""
import sys,pickle,pandas as pd,numpy as np
with open(sys.argv[1],'rb') as f:b=pickle.load(f)
x=pd.read_csv(sys.argv[2]);s=pd.read_csv(sys.argv[3]);p=b['predict'](b,x)
assert s.ID.equals(x.ID) and np.array_equal(b['classes'][p.argmax(axis=1)],s.SUBCLASS.to_numpy())
assert np.isfinite(p).all() and np.allclose(p.sum(axis=1),1)
assert np.allclose(b['predict'](b,x.iloc[:8]),p[:8])
np.save(sys.argv[4],p)
"""
env=os.environ.copy();env.pop('PYTHONPATH',None)
with tempfile.TemporaryDirectory() as td:
 pf=Path(td)/'probs.npy'
 check=subprocess.run([sys.executable,'-c',program,str(model_path9.resolve()),str((DATA/'test.csv').resolve()),str((OUT/'submission_v9.csv').resolve()),str(pf)],cwd=td,env=env,capture_output=True,text=True)
 if check.returncode:raise RuntimeError(check.stderr)
 assert np.allclose(np.load(pf),test_probability9)
proof9=dict(status='passed',version=9,accepted_new_candidate=accepted9,winner=winner9,component_names=NAMES9,reference_weights=W6.tolist(),final_weights=weights9.tolist(),finalists=FINALISTS9,objectives=OBJECTIVES9,weight_plans=len(PLANS9),primary=rank9.loc[winner9].to_dict(),baseline=rank9.loc['v6_reference'].to_dict(),confirmation_seeds=CONFIRM_SEEDS,confirmation=confirmation9.reset_index().to_dict(orient='records') if len(confirmation9) else [],confirmation_deltas=deltas9,passes=passes9,holdout=metrics9,model_bytes=model_path9.stat().st_size,active_models=len(selected_models9),input_slots=int(sum(s['inputs'] for s in sizes9)),cached_component_names=cache_models9,full_fit_reused=full_fit_reused9,full_cache_sha256=full_cache_sha9,full_independent_test_reload='passed',csv_exact_match=True,batch_invariance=True,public_score=None,python=sys.version,data_sha256={f:hashlib.sha256((DATA/f).read_bytes()).hexdigest() for f in ['train.csv','test.csv','sample_submission.csv']},packages={k:importlib.metadata.version(k) for k in ['numpy','pandas','scipy','scikit-learn','lightgbm','xgboost','cloudpickle','matplotlib','nbformat','ipython','threadpoolctl']})
(OUT/'verification.json').write_text(json.dumps(proof9,indent=2))
(WORK/'requirements-v9.txt').write_text('# Python '+platform.python_version()+'\n'+'\n'.join(k+'=='+v for k,v in proof9['packages'].items())+'\n')

def mdtable9(frame):
 lines=['| '+' | '.join(map(str,frame.columns))+' |','| '+' | '.join(['---']*len(frame.columns))+' |']
 for row in frame.itertuples(index=False,name=None):lines.append('| '+' | '.join(f'{x:.6f}' if isinstance(x,(float,np.floating)) else str(x) for x in row)+' |')
 return '\n'.join(lines)
importance9=[]
for m,w in zip(selected_models9,selected_weights9):
 if hasattr(m.est,'feature_importances_'):values=m.est.feature_importances_;kind='tree_importance'
 elif hasattr(m.est,'coef_'):values=np.max(np.abs(m.est.coef_),axis=0);kind='max_absolute_coefficient'
 else:values=np.linalg.norm(m.est.coefs_[0],axis=1);kind='first_layer_weight_norm'
 for name,value in zip(m.names,values):importance9.append(dict(model=m.name,view=m.view,feature=name,importance=float(value),metric=kind,ensemble_weight=float(w)))
importance9=pd.DataFrame(importance9);importance9.to_csv(OUT/'final_feature_importance.csv',index=False)
meanings9={'gene':'유전자 변이의 가중 존재','variant':'정확한 정규화 단백질 변이','position':'유전자와 단백질 위치','type':'유전자별 변이 유형','aa':'유전자와 아미노산 변화 종류','supported':'학습에서 지지된 비동의 변이 존재','numeric':'원시·가중 변이 부담 로그 또는 유형별 개수·비율','module':'명시한 유전자 목록의 가중 변이 집계','missing':'결측 표기'}
importance9['meaning']=importance9.feature.map(lambda f:meanings9.get(f.split(':')[0],'기존 구성 모델의 노트북 정의 참조'))
importance9.to_csv(OUT/'final_feature_dictionary.csv',index=False)
(WORK/'selected_variables_v9.md').write_text('# v9 최종 입력 변수\n\nv6 구성 모델의 변수·전처리·선별·학습 파라미터는 그대로 유지하며 모델 선택/확률 가중치만 바꿉니다. 모델을 제거하면 그 모델의 입력 슬롯은 사라지지만 남은 모델을 새 변수로 재선별하지 않습니다. 입력 중요도 수치는 다른 모델 사이에서 직접 비교하거나 인과효과로 해석하지 않습니다.\n\n'+mdtable9(importance9)+'\n')
basefamilies9={'gene','variant','position','type','aa','supported','missing'}
links9=importance9.copy();links9['gene']=links9.feature.map(lambda f:f.split(':')[1] if f.split(':')[0] in basefamilies9 else '')
counts9=links9[links9.gene.ne('')].groupby('gene').size();views9=links9[links9.gene.ne('')].groupby('gene').view.agg(lambda v:', '.join(sorted(set(v))))
mut9=(train[genes].notna()&train[genes].ne('WT'));testmut9=(test[genes].notna()&test[genes].ne('WT'))
gdict9=pd.DataFrame(dict(gene=genes,train_mutated=mut9.sum().values,train_rate=mut9.mean().values,test_rate=testmut9.mean().values))
gdict9['selected_model_feature_uses']=gdict9.gene.map(counts9).fillna(0).astype(int);gdict9['selected_views']=gdict9.gene.map(views9).fillna('-');gdict9.to_csv(OUT/'gene_dictionary_v9.csv',index=False)
(WORK/'gene_dictionary_v9.md').write_text('# v9 전체 유전자 컬럼 사전\n\n관측 통계이며 유전자 기능·병원성 주석은 원본에 없어 추정하지 않습니다. test_rate는 기술통계로만 사용했습니다. 선택 연결 변수 수는 모델 간 중복을 포함하며 0이어도 숫자·경로 요약에 기여할 수 있습니다.\n\n'+mdtable9(gdict9)+'\n')
weights_table9=pd.DataFrame(dict(model=NAMES9,v6_weight=W6,v9_weight=weights9,inputs=WIDTH9));weights_table9['removed']=weights_table9.v9_weight.eq(0);weights_table9.to_csv(OUT/'final_weight_comparison.csv',index=False)
previous_csv9=SUBMIT/'혜진_submission_v6.csv'
changed9=int((pd.read_csv(previous_csv9).SUBCLASS!=submission9.SUBCLASS).sum()) if previous_csv9.exists() else None
caption9='채택한 v9' if accepted9 else '유지 모델 v6'
confirmation_text9=mdtable9(confirmation9.reset_index()) if len(confirmation9) else '1차 기준 통과 후보가 없어 반복 재분할을 실행하지 않았습니다.'
report9=f'''# 혜진 v9: 앙상블 구성·제거·가중치 비교

## 결론
새 변수·새 학습기 없이 v6의 8개 구성 모델과 전처리·선별·학습 설정을 고정했습니다. 고정 가중치 계획 {len(PLANS9)}개를 비교하고, 최대 2개 후보를 고정해 seed=109/211 반복 그룹 CV로 재학습했습니다. 최종 `{winner9}`, 신규 채택 **{accepted9}**입니다.

{caption9} 기존 홀드아웃 Macro F1 {metrics9['v9_final']['macro_f1']:.6f}, 정확도 {metrics9['v9_final']['accuracy']:.6f}. 기준 v6는 같은 재평가에서 Macro F1 {metrics9['v6_reference']['macro_f1']:.6f}, 정확도 {metrics9['v6_reference']['accuracy']:.6f}입니다. 이 홀드아웃은 이전에도 관측했으므로 독립 최종 성능이 아닙니다. Public은 미확인입니다.

## 무엇을 비교했나
초기 40개 단독/그룹 변경을 본 뒤, 여러 비율을 동시에 바꾸는 Dirichlet 64개(중심 W6, 농도 24/64, seed=904; 홀수 샘플의 0.03 미만 가중치는 0)를 추가했습니다. 총 104개로 적응 탐색의 편향이 있으며 반복 확인 전 탐색 규칙과 후보를 고정했습니다. 한 구성 모델의 상대 비중 0/0.5/1.5/2배, 전체/신규 균등, legacy 제거, 트리 그룹 비중 0.75/1.25배, 트리만/비트리만을 사전에 정해 비교했습니다. 확률 가중치는 합이 1이 되게 정규화하고 같은 가중치 계획은 제거했습니다. 0은 모델 제거를 뜻합니다. 모든 후보가 새로 학습한 별도 분류기는 아니며 같은 구성별 OOF를 서로 다른 비율로 합한 앙상블 후보입니다.

모델 단독 F1이 낮다는 이유만으로 제거하지 않고, 제거 후 앙상블 변화와 오류 겹침을 함께 기록했습니다. 변수 전체를 다시 선택하거나 test 암종 비율을 맞추지 않았습니다. 구성 분류기의 하이퍼파라미터는 v6와 동일합니다.

## 선택 규칙과 확인
원본 중심 후보는 개발 clean +0.003 이상, 스트레스 하락 0.003 이하, 종합 비하락을 요구합니다. 종합 중심 후보는 종합 +0.002 이상, clean 하락 0.002 이하를 요구합니다. 종합 점수는 기존 0.6 clean +0.3 희귀 변이 +0.1 중복 주석입니다. 종합 후보의 최고점과 0.0005 이내라면 모델 수가 적은 구성을 우선합니다.

최대 2개 후보 `{FINALISTS9}`와 비율을 고정한 뒤 두 seed에서 8개 구성 모델을 모두 다시 학습했습니다. 각 seed 평균 clean 하락 0.002 이하, 스트레스 하락 0.005 이하를 요구합니다. 두 seed 평균에서 clean 후보는 clean +0.0015 이상·종합 비하락, robust 후보는 clean +0.0005 이상·종합 +0.001 이상이어야 합니다. 개발 단계의 clean 우선 순서를 유지하며 확인 결과를 보고 새 가중치를 다시 만들지 않습니다.

이 규칙은 통계적 유의성 검정이나 Public 보장이 아닙니다. 모든 분할은 같은 개발 환자를 재사용하며, 개발 CV와 기존 홀드아웃은 여러 버전에서 관측했습니다. 확인 결과에 따른 최종 채택도 선택 과정의 일부여서 완전히 새로운 독립 평가가 아닙니다. 합성 스트레스는 실제 test 분포와 같지 않습니다.

## 개별 구성 모델 성능
{mdtable9(component_scores9)}

## 하나씩 제거한 결과
`delta_`는 같은 조건 v6와의 차이입니다. 행의 모델 이름은 제거한 모델입니다.

{mdtable9(ablation9.reset_index(names='removed_model'))}

오류 중복은 error_overlap.csv에 agreement, both_wrong, 한쪽만 맞힌 비율로 기록했습니다. 모두 개발 OOF에 기반하며 test 정답을 쓰지 않습니다.

## 개발 후보 전체 비교
{mdtable9(rank9.reset_index(names='plan'))}

## 반복 그룹 검증
{confirmation_text9}

고정 후보별 통과 여부: `{passes9}`. 두 seed 평균 차이와 seed별 차이는 selection.json과 verification.json에 있습니다. 기준을 통과하지 않으면 v6를 유지합니다.

## 최종 가중치와 비용
{mdtable9(weights_table9)}

활성 모델 {len(selected_models9)}개, 입력 슬롯 합 {sum(s['inputs'] for s in sizes9):,}개, 모델 {model_path9.stat().st_size/1024**2:.2f} MiB입니다. 슬롯 합은 고유 유전자 수가 아닙니다. v6 대비 test 라벨 변경 {changed9}개는 정답 증가 수가 아닙니다.

## 캐시와 데이터 분리
v5 신규 4개와 v6의 넓은 LGB의 완료 OOF는 원본 데이터 해시·ID·프로필·클래스·분할을 검증한 경우 재사용합니다. legacy 3개는 모든 개발 fold에서 실제 재학습했고, 8개 확률을 원래 v6 비율로 합한 값이 v6 전체 OOF와 일치하는지 확인했습니다. 반복 검증의 구성 모델은 모두 해당 fold에서 새로 학습합니다.

전체 train 모델 캐시는 최종 비율을 고정한 뒤 전체 학습 단계에서만 재사용했습니다. CV·반복 검증·홀드아웃에 전체 train 모델을 쓰지 않았습니다. 캐시가 없거나 CACHE_COMPONENTS=False면 노트북에서 다시 학습합니다. 원본/모델 해시와 환경은 verification.json에 저장했습니다. 다중 작업은 서로 다른 학습 객체를 사용하며 CONFIRM_WORKERS=1로 순차 실행할 수도 있습니다.

## 파일·재현
cencer_v9.ipynb 하나에 EDA·파생변수·선별·학습·검증·저장을 포함했습니다. 프로젝트 Python 파일이나 이전 피클은 필수가 아닙니다. 원본 data/와 requirements-v9.txt 환경은 필요합니다. 피클을 프로젝트 코드 없는 임시 폴더에서 로딩해 전체 test 2,546명의 CSV·ID·확률·배치 일치를 확인했습니다.

최종 피클 artifacts/v9/cancer_model_v9.pkl, 전체 입력 selected_variables_v9.md, 원본 통계 gene_dictionary_v9.md에 기록했습니다. 채택 시 submission/혜진_submission_v9.csv를 만들고 주 노트북을 갱신합니다. 미채택 시 v6를 유지하고 artifacts/v9의 재현용 동일 CSV를 새 제출로 배포하지 않습니다. artifacts는 기존 Git 제외 정책을 유지합니다.
'''
(WORK/'analysis_report_v9.md').write_text(report9)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(10,6));xx=np.arange(8);ax.barh(xx,ablation9.delta_clean);ax.set_yticks(xx,NAMES9);ax.axvline(0,color='black',lw=.8);ax.set_xlabel('Change in clean macro F1 after removing one member');fig.tight_layout();fig.savefig(OUT/'leave_one_out_v9.png',dpi=150);plt.close(fig)
p=WORK/'analysis_report_v9.md';p.write_text(p.read_text()+'\n![Component removal](artifacts/v9/leave_one_out_v9.png)\n')
print('Independent test replay, feature dictionaries and report saved.',flush=True)


Independent test replay, feature dictionaries and report saved.


Matplotlib is building the font cache; this may take a moment.


## 추가 진단용 반복 검증
주 모델 기준은 유지하고, 원본 지표와 합성 시험의 상충을 진단합니다. 개발 결과를 본 뒤 추가한 별도 탐색으로 주 모델 채택과 구분합니다.

In [13]:
# Additional diagnostic branch: primary adoption rules stay unchanged.
diagnostic_names9=list(dict.fromkeys([str(rank9.sort_values('clean',ascending=False).index[0]),str(rank9.sort_values('score',ascending=False).index[0])]))
diagnostic_names9=[name for name in diagnostic_names9 if name!='v6_reference']
(OUT/'diagnostic_frozen_plans.json').write_text(json.dumps(dict(candidates=diagnostic_names9,weights={name:PLANS9[name].tolist() for name in diagnostic_names9},seeds=CONFIRM_SEEDS,criteria='experimental only: per-seed clean delta >= -0.001, pooled clean delta >= 0.0015, pooled noise delta >= -0.015'),indent=2))
saved_finalists9=FINALISTS9.copy();FINALISTS9=diagnostic_names9
rows_d9=[];prob_d9={seed:np.zeros((8,3,len(dev),NC),np.float32) for seed in CONFIRM_SEEDS}
try:
 tasks_d9=[]
 for seed in CONFIRM_SEEDS:
  for fold,(a,b) in enumerate(StratifiedGroupKFold(3,shuffle=True,random_state=seed).split(X.iloc[dev],y[dev],groups[dev])):
   assert not set(groups[dev[a]]).intersection(groups[dev[b]])
   tasks_d9.append((seed,fold,a,b))
 with ThreadPoolExecutor(max_workers=CONFIRM_WORKERS) as pool:
  futures_d9=[pool.submit(confirmation_task9,*task) for task in tasks_d9]
  for future in as_completed(futures_d9):
   seed,fold,b,probability,rows,seconds=future.result();prob_d9[seed][:,:,b]=probability;rows_d9.extend(rows)
   pd.DataFrame(rows_d9).sort_values(['seed','fold','model']).to_csv(OUT/'diagnostic_repeated_cv.csv',index=False)
   np.savez_compressed(OUT/'diagnostic_component_probabilities.npz',dev=dev,labels=labels,names=np.array(NAMES9),**{f'seed_{key}':value for key,value in prob_d9.items()})
   print('Diagnostic repeated CV',seed,'fold',fold,'seconds',round(seconds,1),rows,flush=True)
finally:FINALISTS9=saved_finalists9
summary_d9=pd.DataFrame(rows_d9).groupby(['seed','model'])[['clean','noise','duplicate','score']].mean()
delta_d9={};passes_d9={}
for name in diagnostic_names9:
 delta=pd.DataFrame([summary_d9.loc[(seed,name)]-summary_d9.loc[(seed,'v6_reference')] for seed in CONFIRM_SEEDS],index=CONFIRM_SEEDS)
 delta_d9[name]=dict(per_seed=delta.to_dict(orient='index'),mean=delta.mean().to_dict())
 passes_d9[name]=bool((delta.clean>=-.001).all() and delta.clean.mean()>=.0015 and delta.noise.mean()>=-.015)
(OUT/'diagnostic_verification.json').write_text(json.dumps(dict(status='passed',exploratory=True,primary_adoption_unchanged=True,candidates=diagnostic_names9,summary=summary_d9.reset_index().to_dict(orient='records'),deltas=delta_d9,passes=passes_d9),indent=2))
print('Diagnostic confirmation results:',passes_d9,flush=True);display(summary_d9)
# A passing diagnostic candidate is experimental, not a replacement for v6.
experimental9=None
passed_names9=[name for name in diagnostic_names9 if passes_d9[name]]
if passed_names9:
 chosen_d9=max(passed_names9,key=lambda name:delta_d9[name]['mean']['clean']);wexp9=PLANS9[chosen_d9]
 h=np.load(OUT/'holdout_component_probabilities.npz',allow_pickle=True)
 assert np.array_equal(h['indices'],hold) and np.array_equal(h['names'],NAMES9)
 hprob=weighted9(h['probabilities'],wexp9);hpred=labels[hprob.argmax(axis=1)]
 hexp9=dict(macro_f1=f1_score(y[hold],hpred,average='macro',zero_division=0),accuracy=accuracy_score(y[hold],hpred),weighted_f1=f1_score(y[hold],hpred,average='weighted',zero_division=0),n=len(hold))
 with open(OUT/'cancer_model_v9.pkl','rb') as f:reference_bundle9=pickle.load(f)
 reference_models9={m.name:m for m in reference_bundle9['models']};assert set(reference_models9)==set(NAMES9)
 exp_models9=[reference_models9[name] for name,w in zip(NAMES9,wexp9) if w>0]
 exp_weights9=wexp9[wexp9>0]
 exp_bundle9=dict(version=9,experimental=True,models=exp_models9,weights=exp_weights9,genes=genes,classes=labels,winner=chosen_d9,holdout_metrics=hexp9,predict=predict_bundle9)
 exp_path9=OUT/'cancer_model_v9_clean_experimental.pkl'
 with exp_path9.open('wb') as f:cloudpickle.dump(exp_bundle9,f,protocol=pickle.HIGHEST_PROTOCOL)
 expp9=predict_bundle9(exp_bundle9,test);assert np.isfinite(expp9).all() and np.allclose(expp9.sum(axis=1),1)
 exp_csv9=sample.copy();exp_csv9.SUBCLASS=labels[expp9.argmax(axis=1)]
 exp_csvpath9=SUBMIT/'혜진_submission_v9_clean_experimental.csv';exp_csv9.to_csv(exp_csvpath9,index=False);exp_csv9.to_csv(OUT/'submission_v9_clean_experimental.csv',index=False)
 p9script="""
import sys,pickle,pandas as pd,numpy as np
with open(sys.argv[1],'rb') as f:b=pickle.load(f)
x=pd.read_csv(sys.argv[2]);s=pd.read_csv(sys.argv[3]);p=b['predict'](b,x)
assert s.ID.equals(x.ID) and np.array_equal(b['classes'][p.argmax(axis=1)],s.SUBCLASS.to_numpy())
assert np.allclose(b['predict'](b,x.iloc[:8]),p[:8])
np.save(sys.argv[4],p)
"""
 with tempfile.TemporaryDirectory() as td:
  pf=Path(td)/'p.npy';env=os.environ.copy();env.pop('PYTHONPATH',None)
  rr=subprocess.run([sys.executable,'-c',p9script,str(exp_path9.resolve()),str((DATA/'test.csv').resolve()),str(exp_csvpath9.resolve()),str(pf)],cwd=td,env=env,capture_output=True,text=True)
  if rr.returncode:raise RuntimeError(rr.stderr)
  assert np.allclose(np.load(pf),expp9)
 original=pd.read_csv(OUT/'submission_v9.csv');changed=int((exp_csv9.SUBCLASS!=original.SUBCLASS).sum())
 experimental9=dict(status='passed',experimental=True,candidate=chosen_d9,weights=wexp9.tolist(),primary_cv=rank9.loc[chosen_d9].to_dict(),repeated_deltas=delta_d9[chosen_d9],holdout=hexp9,primary_adopted=False,public_score=None,changed_predictions_vs_v6=changed,independent_reload='passed',csv_exact_match=True,batch_invariance=True,model_bytes=exp_path9.stat().st_size)
 (OUT/'experimental_verification.json').write_text(json.dumps(experimental9,indent=2))
 print('Experimental clean-priority model saved:',chosen_d9,'holdout:',hexp9,'changed:',changed,flush=True)
section='\n## 추가 진단: 원본 F1 우선 후보와 종합 최고 후보의 반복 검증\n\n'
section+='기존 104개 계획은 원래 주 모델 채택 기준을 통과하지 못했습니다. 아래는 그 기준을 낮춰 주 모델을 바꾼 결과가 아니라, 합성 스트레스 조건과 원본 지표의 상충을 살피기 위한 추가 진단입니다. 원본 F1 최고/종합 점수 최고 후보 두 개를 개발 결과에서 고정한 뒤 seed=109/211에서 구성 모델을 모두 새로 학습했습니다. 새 가중치는 만들지 않았습니다. 후보를 본 뒤 추가한 적응 탐색이며 같은 환자 재분할이라 독립 검증이 아닙니다.\n\n'
section+=mdtable9(summary_d9.reset_index())+'\n\n'
section+='실험용 저장 기준은 각 seed 평균 원본 F1 하락 0.001 이하, 두 seed 평균 원본 F1 +0.0015 이상, 평균 합성 스트레스 하락 0.015 이하입니다. 이 기준은 진단을 실행하기 전에 고정했으며 주 모델 기준과 다릅니다. 낮은 합성 스트레스 성능이나 Public 불확실성은 남습니다.\n\n'
section+=f'후보별 실험 기준 통과: `{passes_d9}`. 주 모델 채택 상태는 `{accepted9}`로 그대로 유지합니다.\n'
if experimental9:
 section+=f"\n실험용 `{experimental9['candidate']}`를 별도 저장했습니다. 기존 홀드아웃 F1 {experimental9['holdout']['macro_f1']:.6f}, 정확도 {experimental9['holdout']['accuracy']:.6f}입니다. 가중치와 실험용 선택은 홀드아웃 결과를 보기 전에 고정했습니다. 이 홀드아웃은 기존에 반복 관측한 값이라 독립 최종 평가가 아닙니다. 파일은 submission/혜진_submission_v9_clean_experimental.csv, 모델은 artifacts/v9/cancer_model_v9_clean_experimental.pkl입니다. v6 대비 라벨 변경 {experimental9['changed_predictions_vs_v6']}개는 추가 정답 수가 아닙니다. Public은 아직 모릅니다.\n"
else:section+='\n두 후보 모두 진단 기준을 넘지 못해 실험용 CSV를 추가하지 않았습니다.\n'
path=WORK/'analysis_report_v9.md';body=path.read_text();marker='\n## 추가 진단: 원본 F1 우선 후보와 종합 최고 후보의 반복 검증\n';path.write_text(body.split(marker)[0]+section)
print('Diagnostic repeated validation and documentation complete.',flush=True)


Diagnostic repeated CV 109 fold 1 seconds 140.9 [{'seed': 109, 'fold': 1, 'model': 'v6_reference', 'clean': 0.438201965060693, 'noise': 0.340747012003311, 'duplicate': 0.4353681998145831, 'score': 0.4086821026188674}, {'seed': 109, 'fold': 1, 'model': 'equal_all', 'clean': 0.4378654632093398, 'noise': 0.3342497052278768, 'duplicate': 0.4363811026477125, 'score': 0.40663229975873816}, {'seed': 109, 'fold': 1, 'model': 'joint_dirichlet_64.0_06', 'clean': 0.4361005257682946, 'noise': 0.3470246800567365, 'duplicate': 0.439010835242373, 'score': 0.40966880300223496}]
Diagnostic repeated CV 109 fold 0 seconds 143.1 [{'seed': 109, 'fold': 0, 'model': 'v6_reference', 'clean': 0.4290683040442191, 'noise': 0.30905629028498743, 'duplicate': 0.42991080757616407, 'score': 0.39314895026964414}, {'seed': 109, 'fold': 0, 'model': 'equal_all', 'clean': 0.4253298949937029, 'noise': 0.30482632116760083, 'duplicate': 0.4288813193795145, 'score': 0.38953396528445344}, {'seed': 109, 'fold': 0, 'model': 'joi

/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0 0 0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/U

clean     noise  duplicate     score
seed model                                                           
109  equal_all                0.428483  0.323136   0.431282  0.397159
     joint_dirichlet_64.0_06  0.430227  0.333913   0.430329  0.401343
     v6_reference             0.431437  0.327462   0.430943  0.400195
211  equal_all                0.440492  0.330848   0.436590  0.407208
     joint_dirichlet_64.0_06  0.434120  0.351660   0.434218  0.409392
     v6_reference             0.435268  0.343813   0.434663  0.407771